# CIFAR-100 2D-ALiBi Training (3 Seeds)

Trains **only 2D-ALiBi** × 3 seeds on CIFAR-100, with the same
hyperparameters as the existing 12 CIFAR models.

**Output**: 3 checkpoints in
`/content/drive/My Drive/pe_experiment/results_cifar100/alibi_2d_seed{42,123,456}/best_model.pth`

**Compute**: ~5-6h per seed on Blackwell → ~15-18h total.
You can train all 3 seeds in a single Colab session (CIFAR is small).

**Prerequisites in Drive root**:
- `apply_2d_alibi_patch.py` (patcher)
- `full_scale_experiment.py` (original training script)
- `cifar100_2d_alibi_only.py` (the fokused training script)


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Install dependencies

In [ ]:
!pip install -q timm tqdm


## 4. Stage scripts into /content/

We copy three things from Drive into the Colab working directory:
1. `apply_2d_alibi_patch.py` (the patcher)
2. `full_scale_experiment.py` (your existing IN-100 training script -- used as the source for patching)
3. `cifar100_2d_alibi_only.py` (the focused 2D-ALiBi training script)

Then we patch full_scale_experiment.py to produce
`full_scale_experiment_v2.py` (with TwoDALiBi class added).


In [ ]:
import os, shutil

DRIVE_BASE = "/content/drive/My Drive/pe_experiment"

scripts_to_copy = [
    ("apply_2d_alibi_patch.py", "/content/drive/MyDrive/apply_2d_alibi_patch.py"),
    ("full_scale_experiment.py", os.path.join(DRIVE_BASE, "full_scale_experiment.py")),
    ("cifar100_2d_alibi_only.py", "/content/drive/MyDrive/cifar100_2d_alibi_only.py"),
]
# Try alternative paths if main paths don't exist
alt_paths = {
    "full_scale_experiment.py": [
        "/content/drive/MyDrive/full_scale_experiment.py",
        "/content/drive/MyDrive/pe_experiment/full_scale_experiment.py",
    ],
    "apply_2d_alibi_patch.py": [
        "/content/drive/MyDrive/pe_experiment/apply_2d_alibi_patch.py",
    ],
    "cifar100_2d_alibi_only.py": [
        "/content/drive/MyDrive/pe_experiment/cifar100_2d_alibi_only.py",
    ],
}

for fname, primary in scripts_to_copy:
    src = primary
    if not os.path.isfile(src):
        for alt in alt_paths.get(fname, []):
            if os.path.isfile(alt):
                src = alt
                break
        else:
            raise FileNotFoundError(f"Cannot find {fname} in any expected location")
    dst = f"/content/{fname}"
    shutil.copy(src, dst)
    print(f"  Copied: {src} -> {dst}")

# Apply the 2D-ALiBi patch
exit_code = os.system(
    "cd /content && python apply_2d_alibi_patch.py "
    "full_scale_experiment.py full_scale_experiment_v2.py")
if exit_code != 0:
    raise RuntimeError(f"Patch failed with exit code {exit_code}")

print("\nReady. Patched script exists:", 
      os.path.isfile("/content/full_scale_experiment_v2.py"))


## 5. Run training (3 seeds sequentially)

CIFAR is small enough that all 3 seeds typically fit in one ~24h Colab
session. The script has resume logic: if a seed's `best_model.pth`
already exists with complete history, it skips that seed.

Streaming output via subprocess shows each epoch live.


In [ ]:
import subprocess
import sys

cmd = "cd /content && python -u cifar100_2d_alibi_only.py"

print("=" * 60)
print("CIFAR-100 2D-ALiBi training (3 seeds)")
print("=" * 60)
print()

process = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end='', flush=True)
    sys.stdout.flush()

exit_code = process.wait()
print("\n" + "=" * 60)
if exit_code == 0:
    print("ALL DONE")
else:
    print(f"Exited with code {exit_code}")
print("=" * 60)


## 6. Status check

In [ ]:
import os, json

RESULTS = "/content/drive/My Drive/pe_experiment/results_cifar100"

print("2D-ALiBi CIFAR-100 training status:")
print("=" * 60)
accs = []
for seed in [42, 123, 456]:
    tag = f"alibi_2d_seed{seed}"
    history_path = os.path.join(RESULTS, tag, "training_history.json")
    ckpt_path = os.path.join(RESULTS, tag, "best_model.pth")
    
    if os.path.isfile(history_path):
        with open(history_path) as f:
            h = json.load(f)
        best = max(h['val_acc'])
        n_epochs = len(h['val_acc'])
        size_mb = os.path.getsize(ckpt_path) / 1e6 if os.path.isfile(ckpt_path) else 0
        print(f"  Seed {seed}: {n_epochs:3d} epochs done, best={best:.2f}%, ckpt={size_mb:.0f} MB")
        if n_epochs >= 300:
            accs.append(best)
    else:
        print(f"  Seed {seed}: not started")

if len(accs) == 3:
    import statistics as st
    print("-" * 60)
    print(f"  Mean: {st.mean(accs):.2f}% ± {st.stdev(accs):.2f}")
    print()
    print(f"  Compare with existing CIFAR-100 1D-ALiBi: [check Drive]")
